# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object (not as dict subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing all entities by their `@id`.

We'll list all Record Sets, their `@id`s, and their Fields (with their field `@id`s).

In [ ]:
# List available record sets and their fields (by @id)
record_set_ids = []
for record_set in dataset.record_sets():
    print(f"Record Set: {record_set.id}")
    record_set_ids.append(record_set.id)
    for field in record_set.fields:
        print(f"  Field: {field.id}")

If desired, inspect a sample of records from a specific record set using its `@id` (update with a chosen record set from above).

In [ ]:
# List sample records from a chosen record set by @id
# Choose the first available record set
if record_set_ids:
    rs_id = record_set_ids[0]
    print(f"\nSample records for Record Set @id='{rs_id}':")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

> **All dataset entities are referenced by their `@id`.**

In [ ]:
# Extract records from all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecord Set {rs_id}: Columns -> {df.columns.tolist()}")
    print(df.head(2))

# Select the first Record Set for detailed EDA
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Main Record Set selected for EDA: {main_rs_id}")
else:
    print("No record set available for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering, normalization, and grouping. Reference all fields by their `@id`.

> Select and update the `numeric_field_id` and optionally `group_field_id` to match the fields present in the main record set.

In [ ]:
# EDA parameters (provide the @id of numeric and group fields from above overviews)
# Replace these values with the actual field @ids from your dataset
numeric_field_id = None
group_field_id = None

# Infer numeric and group fields by checking dtypes or name heuristics (as an example)
df = dataframes.get(main_rs_id)
if df is not None and not df.empty:
    # Try to auto-select a numeric field
    for col in df.columns:
        if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try to select a group field (string/categorical)
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    print(f"Auto-detected numeric field for EDA: {numeric_field_id}")
    print(f"Auto-detected group field for grouping: {group_field_id}")
else:
    print("No records available for EDA.")

# Safeguard for absence of numeric fields
if numeric_field_id and df[numeric_field_id].dtype.kind in 'biufc':
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where '{numeric_field_id}' > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field suitable for EDA found in main record set.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship with the group field, if applicable.

For illustration, this cell creates histograms and boxplots for the selected numeric field, grouping by group field if detected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and df is not None:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use `mlcroissant` to:
- Load FAIR² dataset metadata and records from a Croissant schema URL.
- Review and explore record sets and fields by their `@id`.
- Extract and analyze data using pandas, referencing fields via `@id`.
- Apply EDA techniques such as filtering, normalization, and grouping.
- Visualize distributions and relationships between fields.

This framework provides a foundation for further data cleaning, analysis, and modeling, all while following Croissant best practices of referencing schema elements by their unique identifier.